In [2]:
!pip install rapidfuzz
import pandas as pd
import numpy as np

from rapidfuzz import process
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 30.4 MB/s eta 0:00:00


In [3]:
from google.colab import files

uploaded = files.upload()

Saving IMDB top 1000.csv to IMDB top 1000.csv


In [7]:
movie_titles = movies["Title"].tolist()

In [6]:
movies = pd.read_csv("IMDB top 1000.csv")

In [8]:
movies.head()
movies.info()
movies.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Unnamed: 0   1000 non-null   int64  
 1   Title        1000 non-null   object 
 2   Certificate  973 non-null    object 
 3   Duration     1000 non-null   object 
 4   Genre        1000 non-null   object 
 5   Rate         1000 non-null   float64
 6   Metascore    712 non-null    float64
 7   Description  1000 non-null   object 
 8   Cast         1000 non-null   object 
 9   Info         1000 non-null   object 
dtypes: float64(2), int64(1), object(7)
memory usage: 78.3+ KB


(1000, 10)

In [9]:
movies = movies[['Title',
                 'Genre',
                 'Description',
                 'Cast']]

In [ ]:
movies.isnull().sum()

,0
Title,0
Genre,0
Description,0
Cast,0


In [10]:
movies = movies.fillna('')

In [11]:
movies["tags"] = (
    movies["Genre"] + " " +
    movies["Description"] + " " +
    movies["Cast"]
)

In [12]:
movies[['Title','tags']].head()

,Title,tags
0,1. The Shawshank Redemption (1994),Drama Two imprisoned men bond over a number of...
1,2. The Godfather (1972),"Crime, Drama The aging patriarch of an organiz..."
2,3. The Dark Knight (2008),"Action, Crime, Drama When the menace known as ..."
3,4. The Godfather: Part II (1974),"Crime, Drama The early life and career of Vito..."
4,5. The Lord of the Rings: The Return of the Ki...,"Action, Adventure, Drama Gandalf and Aragorn l..."


In [13]:
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies["tags"])

In [14]:
tfidf_matrix.shape

(1000, 5211)

In [15]:
similarity = cosine_similarity(tfidf_matrix)

In [16]:
similarity.shape

(1000, 1000)

In [17]:
def recommend(movie_name):

    result = process.extractOne(
        movie_name,
        movie_titles
    )

    if result is None:
        print("Movie not found!")
        return

    best_match = result[0]
    score = result[1]

    if score < 60:
        print("Movie not found!")
        return

    movie_index = movies[movies["Title"] == best_match].index[0]

    similarity_scores = list(enumerate(similarity[movie_index]))

    sorted_movies = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )[1:6]

    print("="*60)
    print("Selected Movie")
    print("="*60)
    print(best_match)

    print("\nTop 5 Recommendations\n")

    for index, sim in sorted_movies:

        print("Movie :", movies.iloc[index]["Title"])

        if "Genre" in movies.columns:
            print("Genre :", movies.iloc[index]["Genre"])

        if "Rate" in movies.columns:
            print("IMDb Rating :", movies.iloc[index]["Rate"])

        print("-"*50)

In [18]:
recommend("The Dark Knight")

Selected Movie
3. The Dark Knight (2008)

Top 5 Recommendations

Movie : 154. Batman Begins (2005)
Genre : Action, Adventure
--------------------------------------------------
Movie : 63. The Dark Knight Rises (2012)
Genre : Action, Adventure
--------------------------------------------------
Movie : 36. The Prestige (2006)
Genre : Drama, Mystery, Sci-Fi
--------------------------------------------------
Movie : 240. Kill Bill: Vol. 1 (2003)
Genre : Action, Crime, Thriller
--------------------------------------------------
Movie : 33. Joker (2019)
Genre : Crime, Drama, Thriller
--------------------------------------------------


In [19]:
recommend("The Godfather")

Selected Movie
2. The Godfather (1972)

Top 5 Recommendations

Movie : 4. The Godfather: Part II (1974)
Genre : Crime, Drama
--------------------------------------------------
Movie : 74. Apocalypse Now (1979)
Genre : Drama, Mystery, War
--------------------------------------------------
Movie : 304. On the Waterfront (1954)
Genre : Crime, Drama, Thriller
--------------------------------------------------
Movie : 304. On the Waterfront (1954)
Genre : Crime, Drama, Thriller
--------------------------------------------------
Movie : 304. On the Waterfront (1954)
Genre : Crime, Drama, Thriller
--------------------------------------------------


In [21]:
movie = input("Enter Movie Name : ")

recommend(movie)

Enter Movie Name : fast and furious
Selected Movie
13. The Good, the Bad and the Ugly (1966)

Top 5 Recommendations

Movie : 116. For a Few Dollars More (1965)
Genre : Western
--------------------------------------------------
Movie : 166. Unforgiven (1992)
Genre : Drama, Western
--------------------------------------------------
Movie : 233. Million Dollar Baby (2004)
Genre : Drama, Sport
--------------------------------------------------
Movie : 49. Once Upon a Time in the West (1968)
Genre : Western
--------------------------------------------------
Movie : 223. Gran Torino (2008)
Genre : Drama
--------------------------------------------------


In [23]:
movie = input("Enter Movie Name : ")

recommend(movie)

Enter Movie Name : Stand by Me
Selected Movie
265. Stand by Me (1986)

Top 5 Recommendations

Movie : 219. Zindagi Na Milegi Dobara (2011)
Genre : Comedy, Drama
--------------------------------------------------
Movie : 264. The Princess Bride (1987)
Genre : Adventure, Family, Fantasy
--------------------------------------------------
Movie : 239. Dil Chahta Hai (2001)
Genre : Comedy, Drama, Romance
--------------------------------------------------
Movie : 340. Her (2013)
Genre : Drama, Romance, Sci-Fi
--------------------------------------------------
Movie : 340. Her (2013)
Genre : Drama, Romance, Sci-Fi
--------------------------------------------------
